# HW 6 — training targets (Oct 25, 2026)

Data: `load_runs("outcome", roles={"train", "val"})`: Full, Hierarchy, Star; rounds with no bot and with an end marker. Every statistic per topology × dataset × role (train, val, and train + val pooled), with a 95% bootstrap CI over groups (log files; 10,000 resamples, seed 573).

1. `success_rate_pct`: % of rounds that reached consensus.
2. `consensus_time_km_median_s`, `_q1_s`, `_q3_s`: Kaplan–Meier time to consensus; failed rounds censored at the time limit (300 s lab, 180 s cloud).
3. `fraught_pct_mean`: mean % of round fraught, forward hold (`fraught_percent_hold`).
4. `committed_pct_mean`: mean % of round in committed splits; `stall_km_median_s`: KM median committed spell; `_k1/_k2/_k3` on Full (h5 main setting: cutoff 0.9, τ = 1 s).
5. `breaker_excess` (Full): (observed − expected minority breakers) / n, as in h5.
6. `end_sitting_pct`: % of player-time at |s| ≥ 0.9, pooled over rounds (HW 4 part B rule).
7. `relapse_pct`: % of ended committed spells followed by a new spell before consensus.

No new definitions: everything reuses `fraught.py`, `h5.py` and `stats.py` (see `src/slider/targets.py`). A CI bound of `inf` means the KM quantile was not reached in more than 2.5% of resamples.

In [1]:
import pandas as pd
from slider import config
from slider import slider_logs as sl
from slider import targets

runs, table = sl.load_runs("outcome", roles={"train", "val"})
pd.crosstab([table["topology"], table["dataset"]], table["role"], values=table["file"], aggfunc="nunique")   # groups per cell

role               train  val
topology  dataset            
Full      CC          14    2
          CL          14    7
Hierarchy CC          14    2
          CL          14    7
          EL          11    6
Star      CC          14    3

Groups per cell (above). CC val has only 2–3 groups with bot-free complete rounds per topology, so its CIs are very rough.

In [2]:
t = targets.train_targets(runs, table)
t.to_csv(config.TRAIN_TARGETS, index=False)
print(len(t), "rows saved to", config.TRAIN_TARGETS.name)

186 rows saved to train_targets.csv


## Full

In [3]:
targets.show(t, "Full")

dataset                                        CC                           \
role                                        train                      val   
stat                                                                         
success_rate_pct            96.77 [89.29, 100.00]  100.00 [100.00, 100.00]   
consensus_time_km_median_s   45.99 [27.60, 50.00]     60.31 [41.72, 60.31]   
consensus_time_km_q1_s       21.07 [17.71, 32.72]     41.72 [38.04, 41.72]   
consensus_time_km_q3_s      80.52 [59.95, 107.36]     63.57 [63.57, 84.29]   
fraught_pct_mean                0.00 [0.00, 0.00]        0.00 [0.00, 0.00]   
committed_pct_mean           15.29 [10.16, 20.37]      10.43 [8.86, 11.48]   
stall_km_median_s               2.18 [1.16, 2.81]        1.17 [1.14, 1.17]   
stall_km_median_s_k1            3.47 [2.67, 6.20]        2.40 [1.14, 4.65]   
stall_km_median_s_k2            1.32 [1.06, 2.65]        2.02 [1.05, 3.28]   
stall_km_median_s_k3            1.14 [0.54, 2.06]        1.04 [0.99, 1.04]   
breaker_excess                  0.11 [0.04, 0.17]       0.20 [-0.10, 0.40]   
end_sitting_pct              73.23 [69.29, 77.08]     70.30 [69.10, 71.24]   
relapse_pct                  76.92 [65.67, 83.67]     73.33 [66.67, 77.78]   

dataset                                                               CL  \
role                                    train+val                  train   
stat                                                                       
success_rate_pct            97.22 [90.90, 100.00]   85.71 [69.05, 97.62]   
consensus_time_km_median_s   45.99 [32.72, 59.95]  77.00 [58.01, 111.42]   
consensus_time_km_q1_s       23.30 [17.97, 38.49]   34.67 [14.62, 65.35]   
consensus_time_km_q3_s      78.19 [63.57, 105.95]   154.09 [101.00, inf]   
fraught_pct_mean                0.00 [0.00, 0.00]      0.00 [0.00, 0.00]   
committed_pct_mean           14.61 [10.10, 19.10]      5.64 [3.01, 8.74]   
stall_km_median_s               2.17 [1.16, 2.67]      2.80 [1.40, 6.30]   
stall_km_median_s_k1            3.27 [2.40, 5.45]      6.40 [3.30, 9.20]   
stall_km_median_s_k2            2.02 [1.15, 2.65]      1.20 [0.80, 1.70]   
stall_km_median_s_k3            1.04 [0.74, 1.60]      0.80 [0.30, 1.00]   
breaker_excess                  0.12 [0.05, 0.19]      0.20 [0.06, 0.32]   
end_sitting_pct              72.82 [69.36, 76.21]   62.41 [59.64, 65.66]   
relapse_pct                  76.47 [66.67, 82.73]   62.26 [45.00, 73.91]   

dataset                                                                    
role                                           val              train+val  
stat                                                                       
success_rate_pct              76.19 [57.14, 90.48]   82.54 [69.84, 92.06]  
consensus_time_km_median_s  112.61 [83.26, 160.36]  89.19 [71.09, 118.70]  
consensus_time_km_q1_s        70.45 [59.31, 89.78]   55.46 [21.00, 70.45]  
consensus_time_km_q3_s        265.26 [123.25, inf]   191.75 [118.70, inf]  
fraught_pct_mean                 0.00 [0.00, 0.00]      0.00 [0.00, 0.00]  
committed_pct_mean              8.22 [4.55, 11.86]      6.50 [4.25, 8.94]  
stall_km_median_s                2.50 [1.30, 5.10]      2.70 [1.70, 4.30]  
stall_km_median_s_k1             5.10 [3.90, 5.50]      5.50 [3.90, 7.00]  
stall_km_median_s_k2             1.20 [0.80, 1.60]      1.20 [0.80, 1.60]  
stall_km_median_s_k3             0.57 [0.40, 1.70]      0.80 [0.40, 1.00]  
breaker_excess                   0.06 [0.01, 0.15]      0.13 [0.06, 0.23]  
end_sitting_pct               57.05 [50.35, 64.78]   60.24 [56.57, 63.91]  
relapse_pct                   72.92 [57.14, 80.70]   67.33 [55.41, 75.00]

## Hierarchy

In [4]:
targets.show(t, "Hierarchy")

dataset                                        CC                         \
role                                        train                    val   
stat                                                                       
success_rate_pct            94.12 [85.29, 100.00]  83.33 [66.67, 100.00]   
consensus_time_km_median_s   32.41 [18.81, 48.02]   47.19 [47.19, 66.59]   
consensus_time_km_q1_s       17.08 [12.69, 25.87]   30.55 [15.54, 30.55]   
consensus_time_km_q3_s      60.50 [41.27, 108.79]   173.99 [173.99, inf]   
fraught_pct_mean              10.10 [5.33, 14.95]   25.76 [20.67, 30.84]   
committed_pct_mean            16.35 [8.80, 24.70]   29.05 [22.08, 36.02]   
stall_km_median_s               3.12 [2.19, 3.95]      3.25 [3.14, 3.25]   
end_sitting_pct              79.02 [73.54, 82.80]   84.00 [82.69, 85.12]   
relapse_pct                  76.29 [63.77, 82.41]   90.00 [83.33, 92.86]   

dataset                                                               CL  \
role                                    train+val                  train   
stat                                                                       
success_rate_pct            92.50 [84.21, 100.00]  95.12 [87.80, 100.00]   
consensus_time_km_median_s   35.33 [22.02, 48.02]   55.30 [39.00, 71.20]   
consensus_time_km_q1_s       17.08 [13.00, 25.87]   21.41 [16.84, 42.08]   
consensus_time_km_q3_s      63.86 [47.19, 108.79]  81.00 [67.74, 158.16]   
fraught_pct_mean              12.45 [7.30, 17.59]    13.73 [6.99, 20.88]   
committed_pct_mean           18.25 [10.94, 25.58]     8.87 [4.58, 13.54]   
stall_km_median_s               3.14 [2.38, 3.95]      1.40 [1.20, 2.00]   
end_sitting_pct              80.17 [75.68, 83.06]   67.96 [61.57, 73.85]   
relapse_pct                  80.29 [70.24, 85.86]   78.81 [52.08, 87.69]   

dataset                                                                   \
role                                          val              train+val   
stat                                                                       
success_rate_pct            95.00 [83.33, 100.00]  95.08 [89.47, 100.00]   
consensus_time_km_median_s   39.24 [22.60, 55.02]   47.06 [36.84, 67.74]   
consensus_time_km_q1_s       22.60 [17.97, 39.24]   22.60 [18.21, 39.00]   
consensus_time_km_q3_s      75.88 [39.46, 124.84]  81.00 [67.74, 146.22]   
fraught_pct_mean               8.32 [2.22, 16.25]    11.96 [6.87, 17.50]   
committed_pct_mean              7.27 [4.70, 9.67]     8.34 [5.33, 11.62]   
stall_km_median_s               1.40 [1.20, 4.50]      1.40 [1.20, 2.00]   
end_sitting_pct              69.92 [64.51, 72.98]   68.55 [63.67, 72.79]   
relapse_pct                  71.05 [43.75, 81.13]   76.92 [58.43, 85.48]   

dataset                                        EL                          \
role                                        train                     val   
stat                                                                        
success_rate_pct            96.97 [90.91, 100.00]   88.89 [77.78, 100.00]   
consensus_time_km_median_s   28.64 [18.38, 42.79]   71.62 [24.66, 134.25]   
consensus_time_km_q1_s        12.80 [8.64, 20.11]    24.50 [12.93, 71.62]   
consensus_time_km_q3_s       58.95 [38.35, 99.48]  201.24 [74.56, 253.35]   
fraught_pct_mean               9.61 [5.11, 14.39]     16.73 [3.03, 35.46]   
committed_pct_mean             8.29 [5.19, 12.01]     14.18 [2.79, 25.70]   
stall_km_median_s               1.41 [1.20, 1.80]       1.80 [1.20, 2.60]   
end_sitting_pct              74.51 [69.24, 79.05]    78.41 [66.39, 85.69]   
relapse_pct                  85.48 [67.31, 91.87]    89.81 [72.50, 94.12]   

dataset                                            
role                                    train+val  
stat                                               
success_rate_pct            94.12 [88.24, 100.00]  
consensus_time_km_median_s   36.28 [20.63, 54.06]  
consensus_time_km_q1_s       13.79 [11.12, 20.63]  
consensus_time_km_q3_s  

## Star

In [5]:
targets.show(t, "Star")

dataset                                          CC                           \
role                                          train                      val   
stat                                                                           
success_rate_pct            100.00 [100.00, 100.00]  100.00 [100.00, 100.00]   
consensus_time_km_median_s     34.28 [28.30, 37.12]     29.16 [16.46, 58.83]   
consensus_time_km_q1_s         22.45 [16.62, 28.30]     16.46 [12.88, 58.83]   
consensus_time_km_q3_s         41.27 [37.12, 64.44]     59.90 [58.83, 90.12]   
fraught_pct_mean                  0.00 [0.00, 0.00]        0.00 [0.00, 0.00]   
committed_pct_mean               9.34 [5.15, 13.72]     16.36 [12.53, 32.98]   
stall_km_median_s                 2.61 [1.42, 3.66]        1.54 [1.36, 3.63]   
end_sitting_pct                64.12 [58.42, 68.53]     64.04 [58.51, 66.39]   
relapse_pct                    45.71 [26.92, 59.18]     61.54 [33.33, 71.43]   

dataset                                              
role                                      train+val  
stat                                                 
success_rate_pct            100.00 [100.00, 100.00]  
consensus_time_km_median_s     34.28 [29.16, 37.87]  
consensus_time_km_q1_s         18.90 [16.46, 26.08]  
consensus_time_km_q3_s         47.77 [37.87, 64.44]  
fraught_pct_mean                  0.00 [0.00, 0.00]  
committed_pct_mean              10.42 [6.71, 14.34]  
stall_km_median_s                 1.63 [1.42, 3.59]  
end_sitting_pct                64.10 [59.50, 67.88]  
relapse_pct                    50.00 [34.29, 60.71]